# Lab 10: Image Segmentation 2, Segment Anything

*Written by Yalda.*

A semantic segmentation network is trained for a fixed list of classes, for example 19 classes in street photos. For
anything else (cells, cats, your own photos) you would need new labels and a new training run.

Today we use **SAM**, the **Segment Anything Model** ([Kirillov et al., 2023](https://arxiv.org/abs/2304.02643)).
It was trained on **1 billion masks**, and it works differently: you do not train it, you **prompt** it. Click on something,
or draw a box around it, and SAM returns its mask. No training today: we use SAM, open it up, and build things with it.

1. **Prompts.** One click gives three possible masks. More clicks or a box pick the right one.
2. **Inside SAM.** What the image encoder, the prompt encoder and the mask decoder each do.
3. **How SAM was trained.** With simulated clicks, placed where its mask is wrong.
4. **Segment everything.** A grid of clicks over the whole photo, keeping only the good masks.
5. **Instance segmentation.** A detector gives the boxes and the names, SAM gives the masks.
6. **Try it yourself.** SAM on your own photo.

**Extra** (in class if you have time, otherwise at home): SAM 2 follows an object through a **video** from one click.

| Part | Time |
|---|---|
| Setup + 1. Prompts | 30 min |
| 2. Inside SAM | 20 min |
| 3. How SAM was trained | 25 min |
| 4. Segment everything | 25 min |
| 5. Instance segmentation | 10 min |
| 6. Try it yourself | 10 min |
| Extra. SAM 2 on a video | 20 min |

For each **TODO**, write your code, then run the check cell below it. If you are stuck, open the hidden **Solution** cell.

## Environment (skip this on Colab)

On Colab only `segment-anything` is missing, and the cell below installs it. To run this notebook on your own machine or a
lab machine, create a conda environment **once** from a terminal:

```bash
conda create -n vc python=3.11 -y && conda activate vc && pip install torch torchvision numpy matplotlib pillow segment-anything jupyter ipykernel && python -m ipykernel install --user --name vc
```

Then pick the **vc** kernel in Jupyter / VS Code (top right). On Windows with an NVIDIA GPU, get the install command
from [pytorch.org](https://pytorch.org/get-started/locally/) instead.

If you already have an environment, the next cell installs anything missing into the kernel you are using.
If it installs something, **restart the kernel** (*Kernel → Restart*), then continue.

In [ ]:
%pip install -q torch torchvision numpy matplotlib pillow segment-anything

In [ ]:
# check your environment
import os
import sys
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"   # Apple GPU: use the CPU for the few operations it does not have
print("python:", sys.version.split()[0], " from", sys.executable)
try:
    import torch
    import torchvision
    import segment_anything
    print("torch:", torch.__version__, " torchvision:", torchvision.__version__)
    print("NVIDIA GPU:", torch.cuda.is_available(), " Apple GPU:", torch.backends.mps.is_available())
except ImportError as e:
    print("ERROR:", e)
    print("Restart the kernel and run this cell again, or pick the vc kernel (see above).")

## Setup

A GPU helps but is not needed: SAM takes about 2 seconds per photo on a laptop CPU.

In [ ]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"   # Apple GPU: use the CPU for the few operations it does not have

import json
import time
import zipfile
import urllib.request
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn.functional as F
import torchvision
from segment_anything import sam_model_registry, SamPredictor

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():   # Apple GPU
    device = "mps"
else:
    device = "cpu"
print("device:", device)

In [ ]:
RAW = "https://raw.githubusercontent.com/arash-mham/visual-computing-I/main/lab-material/"

def download(url, filename):
    if not os.path.exists(filename):
        print("downloading", filename, "...")
        urllib.request.urlretrieve(url, filename)

# SAM's weights (the smallest SAM, "ViT-B", 375 MB) and 24 photos with hand-drawn outlines (1.5 MB)
download("https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth", "sam_vit_b.pth")
download(RAW + "lab10/coco_instances.zip", "coco_instances.zip")
if not os.path.exists("coco_instances"):
    zipfile.ZipFile("coco_instances.zip").extractall(".")

sam = sam_model_registry["vit_b"](checkpoint="sam_vit_b.pth").to(device).eval()
predictor = SamPredictor(sam)      # a small wrapper around the model that takes care of resizing
print("SAM is ready")

## 1. Prompts

### The photos

We use 24 photos from [COCO](https://cocodataset.org). For every photo, people drew
the outline of each big object by hand. SAM never needs these outlines. We keep them as the **ground truth**, to measure
how good SAM's masks are.

A mask is an array of `True` / `False`, one value per pixel, the same size as the photo.

In [ ]:
with open("coco_instances/objects.json") as f:
    PHOTOS = json.load(f)


def load_photo(i):
    # returns the photo (H, W, 3), the true masks (N, H, W) of its N objects, and their N names
    image = np.array(Image.open(f"coco_instances/images/{i:02d}.jpg"))
    ids = np.array(Image.open(f"coco_instances/masks/{i:02d}.png"))    # 0 = background, 1 = first object, 2 = second, ...
    names = PHOTOS[i]["objects"]
    gt = np.stack([ids == k + 1 for k in range(len(names))])
    return image, gt, names


COLORS = np.array([[30, 144, 255], [255, 140, 0], [50, 205, 50], [220, 20, 60], [148, 0, 211],
                   [0, 206, 209], [255, 215, 0], [255, 105, 180]])


def show(ax, image, masks=(), points=None, labels=None, box=None, title=""):
    # draws the photo, every mask in its own color, clicks as circles (green = positive, red = negative), and a box
    out = image.astype(float)
    for k, m in enumerate(masks):
        out[m] = 0.45 * out[m] + 0.55 * COLORS[k % len(COLORS)]
    ax.imshow(out.astype(np.uint8))
    if points is not None:
        for (x, y), l in zip(points, labels):
            ax.plot(x, y, "o", color="green" if l == 1 else "red", markersize=10, markeredgecolor="black")
    if box is not None:
        x0, y0, x1, y1 = box
        ax.plot([x0, x1, x1, x0, x0], [y0, y0, y1, y1, y0], color="yellow", linewidth=2)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])


fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, i in zip(axes, [0, 4, 9, 15]):
    image, gt, names = load_photo(i)
    show(ax, image, gt, title=f"photo {i}: " + ", ".join(names))
plt.tight_layout()
plt.show()

### One click

Using SAM takes two steps:

1. `predictor.set_image(image)`: SAM looks at the photo. This is the slow step, and it happens **once** per photo.
2. `predictor.predict(...)`: give a prompt, get masks back. This is fast, so you can ask again and again.

A click is a point `(x, y)` plus a label: **1** means "this is part of what I want".
Let's click on the vest of the worker in the middle.

In [ ]:
image, gt, names = load_photo(0)
predictor.set_image(image)

point = np.array([[215, 150]])     # (x, y): x goes right, y goes down
label = np.array([1])
masks, scores, low_res = predictor.predict(point_coords=point, point_labels=label, multimask_output=True)
print("masks:", masks.shape, masks.dtype, "  scores:", scores.round(2))

fig, axes = plt.subplots(1, 4, figsize=(18, 3.6))
show(axes[0], image, points=point, labels=label, title="the click")
axes[0].set_xticks(range(0, image.shape[1], 100))
axes[0].set_yticks(range(0, image.shape[0], 100))
for k in range(3):
    show(axes[k + 1], image, [masks[k]], point, label, title=f"mask {k}   score {scores[k]:.2f}")
plt.tight_layout()
plt.show()

One click, **three answers**: the whole worker, the vest, a piece of the vest. A single click is **ambiguous**: SAM cannot
know which one you meant, so it returns all three and lets you choose.

The **score** is SAM's own guess of how good each mask is (it predicts the IoU of its mask with the truth, without seeing
the truth).

**TODO 1:** to measure a mask we need the **IoU** (Intersection over Union), now between two
True/False masks. Write `mask_iou(a, b)`. If both masks are empty, return 0.

In [ ]:
def mask_iou(a, b):
    # a, b: True/False arrays of the same shape. Returns the IoU, a number between 0 and 1
    # TODO
    intersection = ...
    union = ...
    return ...

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def mask_iou(a, b):
    # a, b: True/False arrays of the same shape. Returns the IoU, a number between 0 and 1
    intersection = (a & b).sum()       # pixels in both masks
    union = (a | b).sum()              # pixels in at least one mask
    return intersection / union if union > 0 else 0.0

In [ ]:
# run this cell to check your code
a = np.array([[True, True, False],
              [False, False, False]])
b = np.array([[False, True, True],
              [False, True, False]])
print("IoU:", mask_iou(a, b))
empty = np.zeros_like(a)
print("correct:", abs(mask_iou(a, b) - 1 / 4) < 1e-9 and mask_iou(a, a) == 1 and mask_iou(a, empty) == 0
      and mask_iou(empty, empty) == 0)

In [ ]:
# gt[0] is the true mask of the worker in the middle
for k in range(3):
    print(f"mask {k}: score {scores[k]:.2f}   IoU with the whole worker {mask_iou(masks[k], gt[0]):.2f}")

The mask with the **highest score** is the vest, not the worker. It is a perfectly good mask of a vest. SAM answered a
different question than the one we had in mind.

### More clicks

To say what you mean, click more. Label **1** is a **positive** click ("this is part of it") and label **0** is a
**negative** click ("this is not part of it"). With several clicks the prompt is no longer ambiguous, so we ask for
one mask: `multimask_output=False`.

**TODO 2:** add one or two clicks so that SAM returns the **whole worker in the middle**. Read the coordinates from the
axes of the first picture above. Goal: IoU of at least **0.85** with at most 3 clicks.

In [ ]:
# TODO: add clicks (x, y) and their labels
points = np.array([[215, 150]])
labels = np.array([1])

masks2, scores2, _ = predictor.predict(point_coords=points, point_labels=labels, multimask_output=False)
fig, ax = plt.subplots(figsize=(6, 4.5))
show(ax, image, [masks2[0]], points, labels, title=f"IoU with the worker: {mask_iou(masks2[0], gt[0]):.2f}")
plt.show()

In [ ]:
# run this cell to check your code
print("clicks:", len(points), "  IoU:", round(float(mask_iou(masks2[0], gt[0])), 3))
print("correct:", len(points) <= 3 and len(points) == len(labels) and mask_iou(masks2[0], gt[0]) >= 0.85)

### A box

The other prompt is a **box** `[x0, y0, x1, y1]` (left, top, right, bottom) around the object. A box is rarely ambiguous.
In a real app the user drags the box, or a detector gives it (Part 5). Here we take the box of the true mask.

**TODO 3:** write `mask_to_box(mask)`. `np.nonzero(mask)` returns the rows (`ys`) and the columns (`xs`) of all `True`
pixels.

In [ ]:
def mask_to_box(mask):
    # mask: (H, W) True/False array. Returns the box around it: np.array([x0, y0, x1, y1])
    ys, xs = np.nonzero(mask)
    # TODO: the smallest and the largest x and y
    x0 = ...
    y0 = ...
    x1 = ...
    y1 = ...
    return np.array([x0, y0, x1, y1])

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def mask_to_box(mask):
    # mask: (H, W) True/False array. Returns the box around it: np.array([x0, y0, x1, y1])
    ys, xs = np.nonzero(mask)
    x0 = xs.min()
    y0 = ys.min()
    x1 = xs.max()
    y1 = ys.max()
    return np.array([x0, y0, x1, y1])

In [ ]:
# run this cell to check your code
toy = np.zeros((6, 8), dtype=bool)
toy[2:5, 1:7] = True
toy[1, 3] = True
print("box:", mask_to_box(toy))
print("correct:", list(mask_to_box(toy)) == [1, 1, 6, 4])

In [ ]:
# the worker on the right (gt[1]), from a box
box = mask_to_box(gt[1])
masks3, scores3, _ = predictor.predict(box=box, multimask_output=False)

fig, ax = plt.subplots(figsize=(6, 4.5))
show(ax, image, [masks3[0]], box=box, title=f"box prompt: IoU {mask_iou(masks3[0], gt[1]):.2f}")
plt.show()

## 2. Inside SAM

SAM has three parts:

```
 photo ─────────► image encoder ─────► image embedding ──────────┐
                  (a big ViT)          256 x 64 x 64             ├──► mask decoder ──► 3 masks + 3 scores
 click or box ──► prompt encoder ────► a few tokens ─────────────┘
                                       256 numbers each
```

- The **image encoder** is a Vision Transformer (ViT). It turns the photo into an **embedding**: a 64×64 grid with 256
  numbers per cell. It does not know about your click.
- The **prompt encoder** turns every click (or box corner) into a **token** of 256 numbers.
- The **mask decoder** is a small Transformer. It looks at the embedding and the tokens together and outputs the masks.

The next cell sends the click from Part 1 through the three parts, one at a time, and shows what goes in and what comes out.

In [ ]:
# one click, step by step (predictor.predict does steps 2 and 3 for you)
predictor.set_image(image)                                    # 1. image encoder: photo -> embedding
embedding = predictor.features                                   # (1, 256, 64, 64)

xy = predictor.transform.apply_coords(point, image.shape[:2])                 # the click, in the resized photo
xy = torch.tensor(xy, dtype=torch.float32, device=device)[None]
lab = torch.tensor(label, dtype=torch.int, device=device)[None]
with torch.no_grad():
    tokens, no_mask = sam.prompt_encoder(points=(xy, lab), boxes=None, masks=None)      # 2. prompt encoder: click -> tokens
    pixel_scores, quality = sam.mask_decoder(                                           # 3. mask decoder: embedding + tokens -> masks
        image_embeddings=embedding, image_pe=sam.prompt_encoder.get_dense_pe(),
        sparse_prompt_embeddings=tokens, dense_prompt_embeddings=no_mask, multimask_output=True)
    pixel_scores = sam.postprocess_masks(pixel_scores, predictor.input_size, predictor.original_size)   # resize to the photo
step_masks = (pixel_scores[0] > 0).cpu().numpy()
quality = quality[0].cpu().numpy()

print(f"1. image encoder    in: photo {image.shape}, resized and padded to (3, 1024, 1024)    out: embedding {tuple(embedding.shape[1:])}")
print(f"2. prompt encoder   in: click {tuple(point[0].tolist())}, label {label[0]}    out: {tokens.shape[1]} tokens of {tokens.shape[2]} numbers (the click, and an empty one)")
print(f"3. mask decoder     in: embedding + tokens    out: {len(step_masks)} masks {step_masks.shape[1:]} and {len(quality)} scores {quality.round(2)}")

# the embedding as a picture: its 256 numbers per cell, squeezed into 3 colors (PCA)
cells = embedding[0].flatten(1).T.cpu().numpy()                     # (4096, 256): one row per cell
cells = cells - cells.mean(axis=0)
colors = cells @ np.linalg.svd(cells, full_matrices=False)[2][:3].T
colors = (colors - colors.min(axis=0)) / (colors.max(axis=0) - colors.min(axis=0))
rows, cols = int(np.ceil(predictor.input_size[0] / 16)), int(np.ceil(predictor.input_size[1] / 16))   # without the padding

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
show(axes[0, 0], image, points=point, labels=label, title="input: a photo and a click")
axes[0, 1].imshow(colors.reshape(64, 64, 3)[:rows, :cols])
axes[0, 1].set_title("1. image encoder: photo → embedding (64×64 cells)")
axes[0, 2].imshow(tokens[0, 0].cpu().reshape(16, 16), cmap="RdBu_r")
axes[0, 2].set_title("2. prompt encoder: click → token (256 numbers)")
for ax in axes[0, 1:]:
    ax.set_xticks([])
    ax.set_yticks([])
for k in range(3):
    show(axes[1, k], image, [step_masks[k]], point, label, title=f"3. mask decoder: mask {k}, score {quality[k]:.2f}")
plt.tight_layout()
plt.show()

- **Image encoder:** the embedding has 256 numbers per cell, so the picture squeezes them into 3 colors. The people, the
  sky and the ground already look different, and the encoder has not seen the click.
- **Prompt encoder:** the token is just 256 numbers. It means "a positive click at this place".
- **Mask decoder:** it combines the two and returns three masks with their scores, the same ones as in Part 1.

How big and how slow is each part?

In [ ]:
for name, part in [("image encoder", sam.image_encoder), ("prompt encoder", sam.prompt_encoder), ("mask decoder", sam.mask_decoder)]:
    n = sum(p.numel() for p in part.parameters())
    print(f"{name:15s} {n:>11,d} weights")

start = time.time()
predictor.set_image(image)                       # runs the image encoder
emb = predictor.features[0].cpu()                # the image embedding
t_encoder = time.time() - start

start = time.time()
for _ in range(10):                              # prompt encoder + mask decoder, 10 times
    predictor.predict(point_coords=point, point_labels=label, multimask_output=True)
t_decoder = (time.time() - start) / 10

print(f"\nimage embedding: {tuple(emb.shape)}")
print(f"set_image: {t_encoder:.2f} s      predict: {t_decoder * 1000:.0f} ms per prompt")

Almost all the weights and almost all the time are in the image encoder, and it runs **once** per photo. After that, every
new click only costs a few milliseconds. This split is what makes SAM interactive: the mask can follow your mouse.

### The image embedding

SAM resizes the photo so that its long side is **1024** pixels, and pads it to a 1024×1024 square. The ViT cuts this
square into patches of **16×16** pixels, so the embedding has 1024 / 16 = **64** cells per side. Every cell holds 256
numbers that describe what its patch looks like.

What did the encoder learn? Pick one cell, and compare its 256 numbers with the 256 numbers of every other cell.
We compare with the **cosine similarity**: 1 means "same direction", 0 means "unrelated".

**TODO 4:** write `similarity_map(emb, x, y, scale)`. `(x, y)` is a pixel of the photo, and `scale` = 1024 / (long side of
the photo), so the pixel is at `(x * scale, y * scale)` in the resized photo.
1. Find the column and the row of the cell that contains the pixel (one cell is 16 pixels wide; use `int(...)`).
2. Take the 256 numbers of that cell: `emb[:, row, col]`.
3. `F.cosine_similarity(emb, v[:, None, None], dim=0)` compares `v` with every cell. `v[:, None, None]` has shape
   (256, 1, 1), so PyTorch repeats it over the 64×64 grid.

In [ ]:
def similarity_map(emb, x, y, scale):
    # emb: (256, 64, 64) tensor. (x, y): a pixel of the photo. Returns a (64, 64) tensor of similarities
    # TODO: the column and the row of the cell that contains the pixel
    col = ...
    row = ...
    # TODO: the 256 numbers of that cell
    v = ...
    # TODO: the cosine similarity of v with every cell
    sim = ...
    return sim

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def similarity_map(emb, x, y, scale):
    # emb: (256, 64, 64) tensor. (x, y): a pixel of the photo. Returns a (64, 64) tensor of similarities
    col = int(x * scale / 16)
    row = int(y * scale / 16)
    v = emb[:, row, col]
    sim = F.cosine_similarity(emb, v[:, None, None], dim=0)
    return sim

In [ ]:
# run this cell to check your code
torch.manual_seed(0)
toy = torch.randn(256, 64, 64)
toy[:, 10, 20] = toy[:, 3, 5]        # cell (row 10, col 20) is a copy of cell (row 3, col 5)
toy[:, 30, 40] = -toy[:, 3, 5]       # and this one is its opposite
s = similarity_map(toy, 43, 26, 2.0)     # pixel (43, 26) with scale 2 is in cell (row 3, col 5)
print("similarity with itself, a copy, the opposite:", round(s[3, 5].item(), 3), round(s[10, 20].item(), 3), round(s[30, 40].item(), 3))
print("correct:", tuple(s.shape) == (64, 64) and abs(s[3, 5].item() - 1) < 1e-4 and abs(s[10, 20].item() - 1) < 1e-4
      and abs(s[30, 40].item() + 1) < 1e-4)

In [ ]:
H, W = image.shape[:2]
scale = 1024 / max(H, W)
rows = int(np.ceil(H * scale / 16))       # the cells that are covered by the photo (the rest is padding)
cols = int(np.ceil(W * scale / 16))

fig, axes = plt.subplots(1, 4, figsize=(18, 3.6))
for ax, (x, y, what) in zip(axes, [(215, 150, "a vest"), (215, 280, "trousers"), (300, 30, "the sky"), (330, 200, "the hydrant")]):
    sim = similarity_map(emb, x, y, scale)[:rows, :cols]
    ax.imshow(image)
    ax.imshow(sim, extent=(0, W, H, 0), cmap="magma", vmin=0, vmax=1, alpha=0.75)
    ax.plot(x, y, "o", color="cyan", markersize=10, markeredgecolor="white")
    ax.set_title(f"cells similar to {what}")
    ax.axis("off")
plt.tight_layout()
plt.show()

**Think:** click on one vest and **all three** vests light up. The embedding tells what each place *looks like*
(vest, trousers, sky), not *which object* it belongs to. Choosing one object is the job of the prompt and the decoder.

### The prompt encoder

The prompt encoder turns every click into a **token**: 256 numbers that say **where** the click is and whether it is
**positive or negative**. Clicks that are close to each other get similar tokens. A box is just two tokens: its top-left
and its bottom-right corner. That is the whole prompt encoder: about 6,000 weights.

### The mask decoder

The mask decoder is a small Transformer. It looks at the click tokens and the image embedding together, and gives every
pixel a **score**: above 0 means "inside the mask". It outputs the three masks from Part 1, and for each mask a guess of
how good it is (the scores you saw in Part 1).

**Think:** SAM predicts the masks at 256×256 and resizes them to the photo. What does that mean for very thin objects
(a wire, a bicycle spoke)?

## 3. How SAM was trained

Nobody clicked a billion times to train SAM. The clicks are **simulated** from the true mask. For every training mask,
SAM plays this game:

1. **First prompt:** a random pixel inside the true mask.
2. SAM predicts a mask, and we compare it with the truth. The **error region** is every pixel where they disagree.
3. **Next click:** pick a random pixel where SAM is wrong.
   - The pixel belongs to the object, but SAM left it out: the click is **positive** ("add this", a green dot in the
     pictures).
   - The pixel does not belong to the object, but SAM painted it: the click is **negative** ("remove this", a red dot).
4. SAM predicts again from all the clicks so far (it also gets its own previous mask as input). Repeat.

After every round the loss compares the mask with the truth, and the weights are updated.

The next cell plays steps 1 and 2 for one object: the car behind the skater in photo 4.

In [ ]:
image, gt, names = load_photo(4)
predictor.set_image(image)
target = gt[0]                                  # the true mask of the car
game_rng = np.random.default_rng(2)

# step 1: the first click is a random pixel of the true mask
ys, xs = np.nonzero(target)
k = game_rng.integers(len(ys))
clicks = [(xs[k], ys[k])]
click_labels = [1]

# step 2: SAM predicts (one click is ambiguous, so we take the mask with the highest score), and we compare with the truth
masks, scores, low_res = predictor.predict(np.array(clicks), np.array(click_labels), multimask_output=True)
best = np.argmax(scores)
pred1 = masks[best]
missed1 = target & ~pred1                       # in the truth, but not in SAM's mask
too_much1 = pred1 & ~target                     # in SAM's mask, but not in the truth

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
show(axes[0], image, [target], clicks, click_labels, title="step 1: the truth, and a random click inside it")
show(axes[1], image, [pred1], clicks, click_labels, title=f"step 2: SAM's mask, IoU {mask_iou(pred1, target):.2f}")
show(axes[2], image, [missed1, too_much1], title="step 2: the error region (blue = missed, orange = too much)")
plt.tight_layout()
plt.show()

The **error region** is everything that is colored in the last picture: the pixels SAM missed, and the pixels it
painted too much.

**TODO 5:** write `next_click(pred, gt, rng)`, step 3 of the game.

In [ ]:
def next_click(pred, gt, rng):
    # pred, gt: (H, W) True/False arrays. Returns a click (x, y) in the error region, and its label (1 or 0)
    # TODO: the error region: True where the prediction and the truth disagree
    error = ...
    ys, xs = np.nonzero(error)
    k = rng.integers(len(ys))          # a random pixel of the error region
    x = xs[k]
    y = ys[k]
    # TODO: 1 if the truth is True at this pixel (SAM missed it), otherwise 0 (SAM painted too much)
    label = ...
    return (x, y), label

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def next_click(pred, gt, rng):
    # pred, gt: (H, W) True/False arrays. Returns a click (x, y) in the error region, and its label (1 or 0)
    error = pred != gt
    ys, xs = np.nonzero(error)
    k = rng.integers(len(ys))          # a random pixel of the error region
    x = xs[k]
    y = ys[k]
    label = int(gt[y, x])
    return (x, y), label

In [ ]:
# run this cell to check your code
rng = np.random.default_rng(0)
truth = np.zeros((5, 6), dtype=bool)
truth[1:4, 1:4] = True
missed = truth.copy()
missed[2, 3] = False                   # the prediction misses the pixel x=3, y=2
too_much = truth.copy()
too_much[0, 5] = True                  # the prediction has one extra pixel at x=5, y=0
c1 = next_click(missed, truth, rng)
c2 = next_click(too_much, truth, rng)
print("missed a pixel:", c1, "  painted too much:", c2)
print("correct:", tuple(c1[0]) == (3, 2) and c1[1] == 1 and tuple(c2[0]) == (5, 0) and c2[1] == 0)

Steps 3 and 4 for the same car, with your `next_click`:

In [ ]:
# step 3: the next click is a random pixel of the error region
click, click_label = next_click(pred1, target, game_rng)
kind = "positive (SAM missed this pixel)" if click_label == 1 else "negative (SAM painted too much here)"

# step 4: SAM predicts again from both clicks (and from its own previous mask)
masks, _, _ = predictor.predict(np.array(clicks + [click]), np.array(click_labels + [click_label]),
                                mask_input=low_res[best][None], multimask_output=False)
pred2 = masks[0]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
show(axes[0], image, [missed1, too_much1], [click], [click_label], title=f"step 3: the next click is {kind}")
show(axes[1], image, [pred2], clicks + [click], click_labels + [click_label],
     title=f"step 4: SAM's new mask, IoU {mask_iou(pred2, target):.2f}")
plt.tight_layout()
plt.show()

`simulate` plays the whole game: step 1, then steps 2 to 4 again and again.

In [ ]:
def simulate(gt, n_clicks, rng):
    # plays the click game for one object (the photo must already be set in the predictor).
    # Returns a list with, for every round: the predicted mask, the clicks so far, their labels
    ys, xs = np.nonzero(gt)
    k = rng.integers(len(ys))
    points = [(xs[k], ys[k])]                  # first click: a random pixel of the object
    labels = [1]
    history = []
    for c in range(n_clicks):
        if c == 0:
            # one click is ambiguous: take the mask with the highest score
            masks, scores, low_res = predictor.predict(np.array(points), np.array(labels), multimask_output=True)
            best = np.argmax(scores)
        else:
            masks, scores, low_res = predictor.predict(np.array(points), np.array(labels),
                                                       mask_input=previous[None], multimask_output=False)
            best = 0
        pred = masks[best]
        previous = low_res[best]               # SAM gets its own last mask as an extra input
        history.append((pred, list(points), list(labels)))
        if (pred != gt).sum() == 0:            # perfect: nothing left to click
            break
        click, label = next_click(pred, gt, rng)
        points.append(click)
        labels.append(label)
    return history


# the whole game with 5 clicks, for the same car
image, gt, names = load_photo(4)
predictor.set_image(image)
history = simulate(gt[0], 5, np.random.default_rng(2))

fig, axes = plt.subplots(1, len(history) + 1, figsize=(3.6 * (len(history) + 1), 3.8))
show(axes[0], image, [gt[0]], title="the truth")
for ax, (pred, points, labels) in zip(axes[1:], history):
    show(ax, image, [pred], points, labels, title=f"{len(points)} click(s): IoU {mask_iou(pred, gt[0]):.2f}")
plt.tight_layout()
plt.show()

Run the cell above a few times with another number in `default_rng(...)`: the first click decides a lot.

Now the same game for **every** object of the 24 photos, to see how the IoU grows with the number of clicks
(about half a minute on a GPU, 1 to 2 minutes on a CPU).

In [ ]:
N_CLICKS = 6
rng = np.random.default_rng(0)
click_ious = []        # one row per object: the IoU after 1, 2, ..., 6 clicks
best_of_three = []     # after 1 click: the best of the three masks (needs the truth, so this is cheating)
box_ious = []          # one box prompt, no clicks

start = time.time()
for i in range(len(PHOTOS)):
    image, gt, names = load_photo(i)
    predictor.set_image(image)
    for g in gt:
        if g.sum() < 1500:                     # skip very small objects
            continue
        history = simulate(g, N_CLICKS, rng)
        ious = [mask_iou(pred, g) for pred, points, labels in history]
        ious = ious + [ious[-1]] * (N_CLICKS - len(ious))
        click_ious.append(ious)

        first = np.array(history[0][1])
        three, _, _ = predictor.predict(first, np.array([1]), multimask_output=True)
        best_of_three.append(max(mask_iou(m, g) for m in three))

        m, _, _ = predictor.predict(box=mask_to_box(g), multimask_output=False)
        box_ious.append(mask_iou(m[0], g))

mean_ious = np.mean(click_ious, axis=0)
print(f"{len(click_ious)} objects, {time.time() - start:.0f} s")
print("mean IoU after 1..6 clicks:", mean_ious.round(2))
print(f"1 click, best of the three masks: {np.mean(best_of_three):.2f}     1 box: {np.mean(box_ious):.2f}")

plt.figure(figsize=(6, 4))
plt.plot(range(1, N_CLICKS + 1), mean_ious, "o-", label="clicks (SAM's favourite mask)")
plt.plot(1, np.mean(best_of_three), "s", color="tab:green", label="1 click, best of the three masks")
plt.axhline(np.mean(box_ious), color="tab:orange", linestyle="--", label="1 box")
plt.xlabel("number of clicks")
plt.ylabel("mean IoU with the truth")
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

What the numbers say:

- **One click is ambiguous.** SAM's favourite mask is often a *part* (a vest, a wheel). But one of the three masks is
  usually right (the green square). In training, only the **best** of the three masks gets the loss, and that is how
  SAM learned to give three different answers.
- **Every click helps**, and a negative click helps as much as a positive one.
- **One box** is worth about three clicks.

**Think:** why does the IoU not reach 1.0? Look closely at "the truth" in the picture above: the outlines were drawn
by people with a few straight lines. SAM's mask is often more exact than the "truth".

**Where did 1 billion masks come from?** From SAM itself, in three stages. (1) People labeled masks with an early SAM
helping them. (2) SAM labeled the easy objects alone, and people added what it missed. (3) The final SAM labeled
11 million photos **alone**: a grid of clicks on every photo, and only the masks it was sure about were kept.
That last step is the next part.

## 4. Segment everything

Nobody clicks, and we still want a mask for every object. The idea: **click everywhere**. Put a regular grid of points
over the photo, use each one as a one-click prompt, and collect all the masks. Then throw away the bad masks and the
duplicates.

**TODO 6:** write `point_grid(n, H, W)`: split the photo into `n` × `n` equal cells, and return the **center** of every
cell as `(x, y)`. A cell is `W / n` wide and `H / n` high.

In [ ]:
def point_grid(n, H, W):
    # returns an (n * n, 2) array with the centers (x, y) of n x n equal cells, row by row
    points = []
    for i in range(n):          # rows (top to bottom)
        for j in range(n):      # columns (left to right)
            # TODO: the center of cell (i, j)
            x = ...
            y = ...
            points.append([x, y])
    return np.array(points)

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def point_grid(n, H, W):
    # returns an (n * n, 2) array with the centers (x, y) of n x n equal cells, row by row
    points = []
    for i in range(n):          # rows (top to bottom)
        for j in range(n):      # columns (left to right)
            x = (j + 0.5) * W / n
            y = (i + 0.5) * H / n
            points.append([x, y])
    return np.array(points)

In [ ]:
# run this cell to check your code
g = point_grid(2, 100, 200)
print(g)
print("correct:", g.shape == (2 * 2, 2) and g.tolist() == [[50, 25], [150, 25], [50, 75], [150, 75]]
      and point_grid(16, 343, 480).shape == (256, 2))

A 16×16 grid gives 256 clicks, and every click gives 3 masks: **768 masks**. Many are bad (a click on the sky, on an
edge between two things) and many are the same object found again and again. SAM filters them with two numbers:

- the **score** from Part 1 (SAM's own guess of the IoU),
- the **stability**. A mask is `scores > 0`. If the object has a sharp border, moving the threshold a little changes
  almost nothing. If the mask is a vague blob, it changes a lot. So we count the pixels with a **stricter** threshold
  (`> +1`) and with a **looser** one (`> -1`) and divide: a stable mask gives a number close to 1.

**TODO 7:** write `stability_score(logits)`. `logits` is a tensor with the scores of `M` masks, shape (M, H, W).
`(logits > t).flatten(1).sum(dim=1)` counts, for every mask, the pixels above `t`.

In [ ]:
def stability_score(logits, offset=1.0):
    # logits: (M, H, W) tensor of mask scores. Returns M numbers between 0 and 1
    # TODO: for every mask, the number of pixels above +offset and above -offset
    strict = ...
    loose = ...
    return strict / loose

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def stability_score(logits, offset=1.0):
    # logits: (M, H, W) tensor of mask scores. Returns M numbers between 0 and 1
    strict = (logits > offset).flatten(1).sum(dim=1)
    loose = (logits > -offset).flatten(1).sum(dim=1)
    return strict / loose

In [ ]:
# run this cell to check your code
sharp = torch.full((4, 4), -10.0)
sharp[1:3, 1:3] = 10.0                 # 4 pixels clearly inside, the rest clearly outside
vague = torch.full((4, 4), -0.5)
vague[1:3, 1:3] = 0.5                  # the same mask, but every pixel is close to 0
s = stability_score(torch.stack([sharp, vague]))
print("sharp mask:", s[0].item(), "  vague mask:", s[1].item())
print("correct:", tuple(s.shape) == (2,) and s[0].item() == 1.0 and s[1].item() == 0.0)

In [ ]:
def segment_everything(image, n=16, min_score=0.88, min_stability=0.95):
    H, W = image.shape[:2]
    predictor.set_image(image)
    points = point_grid(n, H, W)
    all_masks, all_scores, all_stability = [], [], []
    for start in range(0, len(points), 64):                                    # 64 clicks at a time
        p = predictor.transform.apply_coords(points[start:start + 64], (H, W))  # to the resized photo
        p = torch.as_tensor(p, dtype=torch.float32, device=device)[:, None]    # (64, 1, 2): 64 prompts of one click
        l = torch.ones(len(p), 1, dtype=torch.int, device=device)              # all clicks are positive
        logits, scores, _ = predictor.predict_torch(p, l, multimask_output=True, return_logits=True)
        logits = logits.flatten(0, 1)                                          # (64 * 3, H, W)
        all_masks.append((logits > 0).cpu())
        all_scores.append(scores.flatten().cpu())
        all_stability.append(stability_score(logits).cpu())
    masks = torch.cat(all_masks).numpy()
    scores = torch.cat(all_scores)
    stability = torch.cat(all_stability)
    n_all = len(masks)

    # 1. keep the masks that SAM is sure about
    keep = ((scores > min_score) & (stability > min_stability)).numpy()
    masks = masks[keep]
    scores = scores[keep]
    n_sure = len(masks)

    # 2. remove the duplicates with non-maximum suppression on the boxes of the masks
    boxes = torch.tensor(np.array([mask_to_box(m) for m in masks]), dtype=torch.float32)
    keep = torchvision.ops.nms(boxes, scores, 0.7).numpy()
    print(f"{n_all} masks -> {n_sure} that SAM is sure about -> {len(keep)} after removing duplicates")
    return masks[keep], scores[keep].numpy()


def show_everything(ax, image, masks, title=""):
    # every mask in a random color, the big ones first so that the small ones stay visible
    rng = np.random.default_rng(1)
    out = image.astype(float)
    for k in np.argsort([-m.sum() for m in masks]):
        out[masks[k]] = 0.35 * out[masks[k]] + 0.65 * rng.integers(0, 255, 3)
    ax.imshow(out.astype(np.uint8))
    ax.set_title(title)
    ax.axis("off")


image, gt, names = load_photo(3)
grid = point_grid(16, image.shape[0], image.shape[1])
start = time.time()
auto_masks, auto_scores = segment_everything(image)
print(f"{time.time() - start:.1f} s")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].imshow(image)
axes[0].plot(grid[:, 0], grid[:, 1], ".", color="lime", markersize=5)
axes[0].set_title("256 clicks")
axes[0].axis("off")
show_everything(axes[1], image, auto_masks, f"{len(auto_masks)} masks")
plt.tight_layout()
plt.show()

Try other photos (`load_photo(0)`, `(15)`, ...), a finer grid (`n=32`, slow on a CPU), or other thresholds
(`min_score=0.7`).

**Think:**
- The car is one mask, and its door, its window and each wheel are masks too. Which one is "the object"? SAM does not
  decide: all of them are valid.
- SAM gives **masks without names**. It does not know that this is a car. "Segment anything" is not "recognize anything".

Naming the masks is the next part.

## 5. Instance segmentation

Three kinds of segmentation:

| | every pixel gets | two cars next to each other |
|---|---|---|
| **semantic** | a class | one "car" blob |
| **instance** | a class **and** an object number | car 1 and car 2 |
| **SAM** | an object number, but no class | mask 1 and mask 2, no names |

SAM gives a mask for every object, but no names. An object **detector** gives a box and a name for every object, but
no masks. Put them together and you have instance segmentation, without training anything: every box from the detector
becomes a prompt for SAM (Part 1).

The next cell does this with a pretrained detector from torchvision (Faster R-CNN, 80 classes, 160 MB).

In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights

det_weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
detector = fasterrcnn_resnet50_fpn(weights=det_weights).eval()      # runs on the CPU: about 1 s per photo
COCO_NAMES = det_weights.meta["categories"]

image, gt, names = load_photo(4)

# 1. the detector: a box and a name for every object
x = torch.from_numpy(image).permute(2, 0, 1).float() / 255          # (3, H, W), values 0..1
with torch.no_grad():
    out = detector([x])[0]
keep = out["scores"] > 0.7                                          # only the objects the detector is sure about
boxes = out["boxes"][keep].numpy()
det_names = [COCO_NAMES[c] for c in out["labels"][keep]]
print(len(boxes), "objects:", det_names)

# 2. SAM: every box is a prompt, and gives one mask
predictor.set_image(image)
sam_masks = [predictor.predict(box=box, multimask_output=False)[0][0] for box in boxes]

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
show(axes[0], image, title="the detector: boxes and names, no masks")
show(axes[1], image, sam_masks, title="the detector's boxes + SAM's masks")
for ax in axes:
    for k, (box, name) in enumerate(zip(boxes, det_names)):
        color = COLORS[k % len(COLORS)] / 255
        x0, y0, x1, y1 = box
        ax.plot([x0, x1, x1, x0, x0], [y0, y0, y1, y1, y0], color=color, linewidth=2)
        ax.text(x0, y0 - 3, name, color="white", fontsize=9, bbox=dict(facecolor=color, edgecolor="none", pad=1))
plt.tight_layout()
plt.show()

**Think:**
- The names come from the detector, so this model knows only its 80 classes. Newer models take **text** as the prompt
  ("the red car"): Grounded-SAM puts a text-to-box detector in front of SAM, and SAM 3 (2025) has text prompts built in.
- If the detector misses an object or gives a wrong box, SAM cannot fix it.

## 6. Try it yourself

Use your own photo: upload it (on Colab: the folder icon on the left) and change the file name. Then choose your clicks.
The last picture cuts the object out, like the "sticker" feature on a phone.

In [ ]:
download(RAW + "lab8/dachshund.jpg", "dachshund.jpg")     # a sample photo
my_image = Image.open("dachshund.jpg").convert("RGB")     # <- your own photo here
my_image.thumbnail((1024, 1024))                          # make it smaller if it is huge
my_image = np.array(my_image)
predictor.set_image(my_image)

plt.figure(figsize=(7, 5))
plt.imshow(my_image)                                      # read your click coordinates from the axes
plt.show()

In [ ]:
my_points = np.array([[480, 330], [330, 450]])     # <- your clicks (x, y)
my_labels = np.array([1, 1])                       # <- 1 = positive, 0 = negative

my_masks, my_scores, _ = predictor.predict(point_coords=my_points, point_labels=my_labels, multimask_output=True)
best = np.argmax(my_scores)               # or choose 0, 1 or 2 yourself

sticker = np.full_like(my_image, 255)     # white background
sticker[my_masks[best]] = my_image[my_masks[best]]

fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for k in range(3):
    show(axes[k], my_image, [my_masks[k]], my_points, my_labels, title=f"mask {k}   score {my_scores[k]:.2f}")
show(axes[3], my_image, title="photo")
show(axes[4], sticker, title=f"sticker from mask {best}")
plt.tight_layout()
plt.show()

More things to try:
- `show_everything` with `segment_everything(my_image)` on your photo.
- A photo that is very different from normal photos: a microscope image, a satellite image, a drawing, an X-ray.
  SAM was trained on everyday photos. Where does it start to fail?

## Extra: SAM 2 on a video

*In class if you have time, otherwise at home.*

A video is a list of photos, so we could run SAM on every frame. But SAM does not know that the boy in frame 10 is the
boy from frame 0. We would have to click again in every frame.

**SAM 2** ([Ravi et al., 2024](https://arxiv.org/abs/2408.00714)) adds a **memory**. It stores the embeddings and the
masks of the frames it has already seen, and when it segments a new frame, the decoder also looks at this memory.
You click **once**, in one frame, and SAM 2 follows the object through the video, even after it was hidden for a while.
The rest (image encoder, prompt encoder, mask decoder) is the same idea as in Part 2.

This part downloads the smallest SAM 2 (156 MB) and a short video from the SAM 2 authors (13 MB).

In [ ]:
%pip install -q sam2

In [ ]:
from sam2.build_sam import build_sam2_video_predictor

download("https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt", "sam2.1_hiera_tiny.pt")
download("https://dl.fbaipublicfiles.com/segment_anything_2/assets/bedroom.zip", "bedroom.zip")
if not os.path.exists("bedroom"):
    zipfile.ZipFile("bedroom.zip").extractall(".")           # 200 frames: bedroom/00000.jpg ... 00199.jpg

# to be fast, keep every 5th frame: 40 frames
os.makedirs("bedroom_small", exist_ok=True)
for k in range(0, 200, 5):
    Image.open(f"bedroom/{k:05d}.jpg").save(f"bedroom_small/{k:05d}.jpg", quality=95)
frame_files = sorted(os.listdir("bedroom_small"))
frames = [np.array(Image.open("bedroom_small/" + f)) for f in frame_files]
print(len(frames), "frames of", frames[0].shape)

video_predictor = build_sam2_video_predictor("configs/sam2.1/sam2.1_hiera_t.yaml", "sam2.1_hiera_tiny.pt", device=device)
state = video_predictor.init_state(video_path="bedroom_small")     # loads the frames

fig, axes = plt.subplots(1, 4, figsize=(18, 3))
for ax, k in zip(axes, [0, 13, 26, 39]):
    ax.imshow(frames[k])
    ax.set_title(f"frame {k}")
    ax.set_xticks(range(0, 960, 200))
plt.tight_layout()
plt.show()

We prompt SAM 2 only in **frame 0**. Every object gets its own number (`obj_id`).

**TODO 8:** the cell below clicks once on the boy, and SAM 2 selects only his **shorts** (the same ambiguity as in
Part 1). Add a second click so that the mask covers the **whole boy**.

In [ ]:
# TODO: add a click
boy_points = np.array([[210, 350]], dtype=np.float32)
boy_labels = np.array([1], dtype=np.int32)

video_predictor.reset_state(state)
_, obj_ids, logits = video_predictor.add_new_points_or_box(state, frame_idx=0, obj_id=1, points=boy_points, labels=boy_labels)
boy_mask = (logits[0, 0] > 0).cpu().numpy()

fig, ax = plt.subplots(figsize=(7, 4))
show(ax, frames[0], [boy_mask], boy_points, boy_labels, title=f"frame 0: {boy_mask.sum()} pixels")
plt.show()

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

boy_points = np.array([[210, 350], [250, 220]], dtype=np.float32)     # the shorts and the shirt
boy_labels = np.array([1, 1], dtype=np.int32)

video_predictor.reset_state(state)
_, obj_ids, logits = video_predictor.add_new_points_or_box(state, frame_idx=0, obj_id=1, points=boy_points, labels=boy_labels)
boy_mask = (logits[0, 0] > 0).cpu().numpy()

fig, ax = plt.subplots(figsize=(7, 4))
show(ax, frames[0], [boy_mask], boy_points, boy_labels, title=f"frame 0: {boy_mask.sum()} pixels")
plt.show()

In [ ]:
# run this cell to check your code
print("pixels in the mask:", boy_mask.sum())
print("correct:", 10000 < boy_mask.sum() < 30000)

In [ ]:
# a second object: the girl (obj_id 2), also only in frame 0
girl_points = np.array([[400, 150], [420, 300]], dtype=np.float32)
girl_labels = np.array([1, 1], dtype=np.int32)
video_predictor.add_new_points_or_box(state, frame_idx=0, obj_id=2, points=girl_points, labels=girl_labels)

# follow both through the video (about a minute without an NVIDIA GPU)
video_masks = {}
for frame_idx, obj_ids, logits in video_predictor.propagate_in_video(state):
    video_masks[frame_idx] = (logits[:, 0] > 0).cpu().numpy()        # (2, H, W): the boy and the girl

fig, axes = plt.subplots(4, 5, figsize=(20, 10))
for ax, k in zip(axes.flatten(), range(0, 40, 2)):
    show(ax, frames[k], video_masks[k], title=f"frame {k}")
plt.tight_layout()
plt.show()

**Think:**
- Around frame 28 the boy jumps off the bed and almost leaves the picture, and later he comes back. How can SAM 2 know it
  is the same boy? (Blue should still be the boy and orange the girl.)
- The two children overlap in several frames. What would happen if we ran SAM on every frame with the box of the
  previous mask as the prompt?

To go further: the [SAM 2 video notebook](https://colab.research.google.com/github/facebookresearch/sam2/blob/main/notebooks/video_predictor_example.ipynb)
from the authors shows how to fix a mistake in the middle of a video with one more click.